# Solving a complex task with a multi-agent hierarchy

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

The reception is approaching! With your help, Alfred is now nearly finished with the preparations.

But now there's a problem: the Batmobile has disappeared. Alfred needs to find a replacement, and find it quickly.

Fortunately, a few biopics have been done on Bruce Wayne's life, so maybe Alfred could get a car left behind on one of the movie set, and re-engineer it up to modern standards, which certainly would include a full self-driving option.

But this could be anywhere in the filming locations around the world - which could be numerous.

So Alfred wants your help. Could you build an agent able to solve this task?

> 👉 Find all Batman filming locations in the world, calculate the time to transfer via a cargo plane to there, and represent them on a map, with a color varying by a cargo plane transfer time. Also represent some supercar factories with the same cargo plane transfer time.

Let's build this!

In [7]:
#!pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q

In [1]:
#from huggingface_hub import notebook_login

#notebook_login()

from smolagents import LiteLLMModel

QWEN25_3B_Q4_K_M = "qwen2.5-coder:3b-instruct-q4_K_M"
QWEN25_7B_Q4_K_M = "qwen2.5-coder:7b-instruct-q4_K_M"
QWEN25_14B_Q4_K_M = "qwen2.5-coder:14b-instruct-q4_K_M"
QWEN25_32B_Q4_K_M = "qwen2.5-coder:32b-instruct-q4_K_M"

QWEN3_752M_Q4_K_M = "qwen3:0.6b"

QWEN3_4B_Q4_K_M = "qwen3:4b"

QWEN3_8B_Q4_K_M = "qwen3:8b"

QWEN3_30B_A3B_Q4_K_M = "qwen3:30b-a3b"

DEEPSEEK_R1_7B = "deepseek-r1:7b"

LLAMA32_VISION_11B_Q4_K_M = "llama3.2-vision:11b-instruct-q4_K_M"

MODEL_ID = "/".join(["ollama_chat", QWEN25_14B_Q4_K_M])

lite_model = LiteLLMModel(
    model_id=MODEL_ID,
    api_base="http://localhost:11434",
)

WEB_MODEL_ID = "/".join(["ollama_chat", QWEN3_4B_Q4_K_M])

web_lite_model = LiteLLMModel(
    model_id=WEB_MODEL_ID,
    api_base="http://localhost:11434",
    max_tokens=8096
)

MA_MODEL_ID =  "/".join(["ollama_chat", DEEPSEEK_R1_7B])

ma_lite_model =  LiteLLMModel(
    model_id=MA_MODEL_ID,
    api_base="http://localhost:11434",
    max_tokens=8096
)

VISION_MODEL_ID = "/".join(["ollama_chat", LLAMA32_VISION_11B_Q4_K_M])

vision_lite_model =  LiteLLMModel(
    model_id=VISION_MODEL_ID,
    api_base="http://localhost:11434",
    max_tokens=8096
)


In [2]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


For the model provider, we use Together AI, one of the new [inference providers on the Hub](https://huggingface.co/blog/inference-providers)!

Regarding the GoogleSearchTool: this requires either having setup env variable `SERPAPI_API_KEY` and passing `provider="serpapi"` or having `SERPER_API_KEY` and passing `provider=serper`.

If you don't have any Serp API provider setup, you can use `DuckDuckGoSearchTool` but beware that it has a rate limit.

In [3]:
import os
from PIL import Image
from smolagents import CodeAgent, VisitWebpageTool


#model = HfApiModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

We can start with creating a baseline, simple agent to give us a simple report.

In [4]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [5]:
#from google.colab import userdata
import os
#os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')

from langchain.agents import load_tools
from smolagents import CodeAgent, Tool

search_tool = Tool.from_langchain(load_tools(["searx-search"], searx_host="http://localhost:8080")[0])


In [6]:
if not ( 'search_tool' in locals() and 'search_tool' in globals() ):
    raise ValueError("Run an above cell to create a search_tool instance")

if not lite_model:
    raise ValueError("Run cell near top of notebook to ensure lite_model instance")

agent = CodeAgent(
    model=lite_model,
    tools=[
        search_tool,
        VisitWebpageTool(),
        calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [14]:
if not ( 'task' in locals() and 'task' in globals() ):
    raise ValueError("Run an above cell to create a task variable")

if not isinstance(task, str):
    raise TypeError("Variable task must be string")

result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen3:4b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern ```(?:py|python)?\s*\n(.*?)\n``` was not found in it.
            Here is your code snippet:
            <think>
Okay, the user wants me to find all Batman filming locations worldwide, calculate the transfer time via cargo plane
from Gotham (40.7128° N, 74.0060° W), and return them as a pandas DataFrame. Also, they want some supercar 
factories with the same transfer time. Let me break this down.

First, I need to find all Batman filming locations. I remember that the main Batman films are "The Dark Knight" 
(2008), "The Dark Knight Rises" (2012), "Batman v Superman: Dawn of Justice" (2016), and "Batman: The Dark Knight 
Returns" (2012). I should check each of these for their filming locations. For example, "The Dark Knight" was 
filmed in New York City, but also in places like Chicago and other cities. "The Dark Knight Rises" was filmed in 
New York, London, and some other locations. "Batman v Superman" was shot in New York, California, and some other 
places. Also, "The Dark Knight Returns" was shot in New York, Canada, and other places. I might need to look up 
each film's specific locations.

Once I have the list of locations, I need to calculate the time it takes to transfer via cargo plane from Gotham to
each of those locations. Since Gotham is in New York City, the coordinates are given as 40.7128° N, 74.0060° W. 
I'll need to convert each filming location's coordinates into latitude and longitude to calculate the distance 
between them. Then, using the distance, I can estimate the flight time. Cargo planes typically fly at around 
500-600 mph, so the time would be distance divided by that speed.

But wait, do I have the exact coordinates for each filming location? Some locations might be cities, so maybe I can
use their approximate coordinates. For example, New York City is around 40.7128° N, 74.0060° W (which is Gotham's 
coordinates), so the distance from Gotham to New York would be zero. But maybe the user wants actual filming 
locations, not just the city. So I need to find the specific filming locations for each movie. For example, "The 
Dark Knight" was filmed in New York City, but also in Chicago, where the "Gotham" set was built. Wait, but the user
is in Gotham, which is a fictional city in the movies, but the real-world filming locations are in real cities. So 
I need to make sure I get the real-world locations.

Next, I need to find the exact coordinates for each of these filming locations. Maybe I can use a database or a 
list of filming locations. For example, "The Dark Knight" was filmed in New York City, Chicago, and other places. I
need to look up each of these. Once I have the coordinates, I can calculate the distance between Gotham and each 
location.

Then, calculate the flight time. For example, if a location is in New York City, the distance is zero, so time is 
zero. If a location is in London, the distance is about 5500 km, so flight time would be 5500 km divided by 500 
mph, which is about 11 hours. But wait, the user is in Gotham, so the distance from Gotham to each filming 
location.

Wait, but the user is in Gotham, which is a fictional city. However, the actual filming locations are in real 
cities. So the coordinates of the real cities are used. So, for example, if a filming location is in New York City,
the distance from Gotham (which is in New York) would be zero. But if a location is in Chicago, the distance would 
be the distance between New York and Chicago.

So, the steps are:

1. List all Batman film locations (each movie's specific filming locations).
2. For each location, get its latitude and longitude.
3. Calculate the distance between Gotham (40.7128° N, 74.0060° W) and each location.
4. Calculate flight time based on distance and average cargo plane speed.
5. Compile this into a pandas DataFrame.

Then, for the supercar factories, find real-world supercar factories, calculate the same distance and time

[Step 1: Duration 912.30 seconds| Input tokens: 83 | Output tokens: 5,923]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 728.25 seconds| Input tokens: 2,131 | Output tokens: 9,542]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(5570.00)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 5570.0

[Step 3: Duration 291.23 seconds| Input tokens: 4,179 | Output tokens: 10,493]

In [15]:
result

5570.0

We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [8]:
if not ( 'task' in locals() and 'task' in globals() ):
    raise ValueError("Run an above cell to create a task variable")

if not isinstance(task, str):
    raise TypeError("Variable task must be string")

if not agent:
    raise ValueError("Run an above cell to create an agent instance")

if not isinstance(agent, CodeAgent):
    raise TypeError("Variable agent must be CodeAgent")

agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:14b-instruct-q4_K_M ──────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts Survey

### 1.1. Facts given in the task
- Current location: Gotham, coordinates (40.7128° N, 74.0060° W)

### 1.2. Facts to look up
- **Batman filming locations**: 
  - Search query: "List of Batman filming locations"
  - Source: IMDb, Wikipedia, or specific movie databases.

- **Supercar factories**:
  - Search query: "List of supercar manufacturers and their locations"
  - Source: Manufacturer websites, automotive industry databases.

- **Cargo plane cruising speed**:
  - Search query: "Average cruising speed for cargo planes"
  - Source: Aviation industry data, manufacturer specifications, or general aviation resources.

### 1.3. Facts to derive
- **Travel time from Gotham to each Batman filming location**:
  - Using `calculate_cargo_travel_time` function with Gotham's coordinates as the origin and the coordinates of 
each filming location as the destination.

- **Travel time from Gotham to each supercar factory**:
  - Similarly, using `calculate_cargo_travel_time` function with Gotham's coordinates as the origin and the 
coordinates of each supercar factory as the destination.

## 2. Plan

1. **Search for Batman filming locations**:
   - Use `searx_search("List of Batman filming locations")` to gather a list of filming locations.

2. **Visit webpages to confirm Batman filming locations**:
   - For each location found, visit the source webpage using `visit_webpage(url)` and confirm the coordinates.

3. **Search for supercar factories**:
   - Use `searx_search("List of supercar manufacturers and their locations")` to gather a list of supercar 
factories.

4. **Visit webpages to confirm supercar factory locations**:
   - For each factory found, visit the source webpage using `visit_webpage(url)` and confirm the coordinates.

5. **Search for cargo plane cruising speed**:
   - Use `searx_search("Average cruising speed for cargo planes")` to find the average cruising speed of cargo 
planes.

6. **Calculate travel time from Gotham to each Batman filming location**:
   - For each confirmed Batman filming location, use `calculate_cargo_travel_time(origin_coords=(40.7128, 
-74.0060), destination_coords=(filming_location_lat, filming_location_lon))` to calculate the travel time.

7. **Calculate travel time from Gotham to each supercar factory**:
   - For each confirmed supercar factory, use `calculate_cargo_travel_time(origin_coords=(40.7128, -74.0060), 
destination_coords=(factory_lat, factory_lon))` to calculate the travel time.

8. **Create a pandas dataframe with Batman filming locations and their travel times**:
   - Organize the data into a pandas dataframe with columns: `Location Name`, `Coordinates (Lat, Lon)`, `Travel 
Time`.

9. **Create a pandas dataframe with supercar factories and their travel times**:
   - Organize the data into a pandas dataframe with columns: `Factory Name`, `Coordinates (Lat, Lon)`, `Travel 
Time`.

10. **Combine both dataframes and return the final answer**:
    - Merge the two dataframes based on travel time.
    - Use `final_answer(combined_dataframe)` to return the combined pandas dataframe.

\n
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_locations_query = "List of Batman filming locations"                                                      
  batman_location_results = searx_search(batman_locations_query)                                                   
  print(batman_location_results)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'batman_location_results = searx_search(batman_locations_query)' due to: ValueError: 
('Searx API returned an error: ', '<!doctype html>\n<html lang=en>\n<title>403 
Forbidden</title>\n<h1>Forbidden</h1>\n<p>You don&#39;t have the permission to access the requested resource. It is
either read-protected or not readable by the server.</p>\n')

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 11 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Combine all travel times into a single dictionary                                                              
  all_travel_times = {                                                                                             
      "Batman Film Locations": travel_times_batman_film_locations,                                                 
      "Supercar Factories": travel_times_supercar_factories                                                        
  }                                                                                                                
                                                                                                                   
  # Find the shortest travel time and corresponding location type and destination                                  
  shortest_time = float('inf')                                                                                     
  shortest_location_type = ""                                                                                      
  shortest_destination = ""                                                                                        
                                                                                                                   
  for category, locations in all_travel_times.items():                                                             
      for destination, (coords, travel_time) in locations.items():                                                 
          if travel_time < shortest_time:                                                                          
              shortest_time = travel_time                                                                          
              shortest_location_type = category                                                                    
              shortest_destination = destination                                                                   
                                                                                                                   
  print(f"Shortest travel time: {shortest_time} hours")                                                            
  print(f"Location type: {shortest_location_type}")                                                                
  print(f"Destination: {shortest_destination}")                                                                    
                                                                                                                   
  # Provide the final answer                                                                                       
  final_answer({                                                                                                   
      "type": shortest_location_type,                                                                              
      "destination": shortest_destination,                                                                         
      "travel_time": shortest_time                                                                                 
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Shortest travel time: 9.12 hours
Location type: Supercar Factories
Destination: McLaren

Out - Final answer: {'type': 'Supercar Factories', 'destination': 'McLaren', 'travel_time': 9.12}

[Step 11: Duration 403.95 seconds| Input tokens: 41,119 | Output tokens: 2,026]

{'type': 'Supercar Factories', 'destination': 'McLaren', 'travel_time': 9.12}


In [9]:
detailed_report

{'type': 'Supercar Factories', 'destination': 'McLaren', 'travel_time': 9.12}

Thanks to these quick changes, we obtained a much more concise report by simply providing our agent a detailed prompt, and giving it planning capabilities!

💸 But as you can see, the context window is quickly filling up. So **if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs**.

➡️ We need to improve the structure of our system.

## ✌️ Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:
- Each agent is more focused on its core task, thus more performant
- Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.

Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `plotly`, and `geopandas` + `shapely` for spatial plotting.

In [ ]:
#model = HfApiModel(
#    "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
#)

if not ( 'search_tool' in locals() and 'search_tool' in globals() ):
    raise ValueError("Run an above cell to create a search_tool instance")

if not web_lite_model:
    raise ValueError("Run cell near top of notebook to ensure web_lite_model instance")

web_agent = CodeAgent(
    model=web_lite_model,
    tools=[
        search_tool,
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1), and add a `planning_interval` to the mix.

In [ ]:
#from google.colab import userdata
#import os
#os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
import os
from smolagents.utils import encode_image_base64, make_image_url
#from smolagents import OpenAIServerModel

if not vision_lite_model:
    raise ValueError("Run cell near top of notebook to ensure vision_lite_model instance")

def check_reasoning_and_plot(final_answer, agent_memory):
    #multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
    multimodal_model = vision_lite_model
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True

if not ma_lite_model:
    raise ValueError("Run cell near top of notebook to ensure ma_lite_model instance")

manager_agent = CodeAgent(
    #model=HfApiModel("deepseek-ai/DeepSeek-R1", provider="together", max_tokens=8096),
    model=ma_lite_model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

Let us inspect what this team looks like:

In [ ]:
manager_agent.visualize()

In [ ]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_map.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

I don't know how that went in your run, but in mine, the manager agent skilfully divided tasks given to the web agent in `1. Search for Batman filming locations`, then `2. Find supercar factories`, before aggregating the lists and plotting the map.

Let's see what the map looks like by inspecting it directly from the agent state:

In [ ]:
manager_agent.python_executor.state["fig"]

![output map](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/smolagents/output_map.png)